# 04 · Modeling agent
Documentation: `docs/agents/modeling_agent.md`.

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_rows', 80); pd.set_option('display.width', 200)
D = ROOT / 'data'
def ok(p):
    if not Path(p).exists(): print('File not available yet:', p); return False
    return True
from core.config import load_config
cfg = load_config(); mcfg = cfg['modeling']
rep = json.loads((D/'modeling/risk_report.json').read_text())
print('Date:', rep['date'])

## 1. Calibrated parameters

In [ ]:
cal = pd.DataFrame(rep['calibration']['params']).T
cal['long_run_vol'] = np.sqrt(cal.theta); cal['vol_today'] = np.sqrt(cal.v0)
display(cal.round(3)); print('CIR yield:', {k: round(v,4) for k, v in rep['calibration']['rate'].items()})
print('Current regime:', 'crisis' if rep['calibration']['crisis_now'] else 'calm', '· share of crisis weeks:', round(rep['calibration']['crisis_share'],2))

## 2. Portfolio factor exposures

In [ ]:
pd.Series(rep['portfolio_beta']).round(3).to_frame('beta').T

## 3. Monte Carlo results (1,000,000 paths)

In [ ]:
pd.DataFrame(rep['monte_carlo']).round(4)

## 4. Stress tests

In [ ]:
display(pd.DataFrame(rep['stress_historical']).T)
display(pd.DataFrame(rep['stress_hypothetical']).T)

## 5. Re-run a simulation here: 3-month factor distribution

In [ ]:
from agents.portfolio_agent.agent import load
from agents.modeling_agent.calibrate import calibrate
from agents.modeling_agent.simulate import factor_terminal
_, pe, _, uni, _ = load(D/'processed', 'EUR'); macro = pd.read_parquet(D/'processed/macro.parquet')
c = calibrate(pe, macro, mcfg)
Fs = pd.DataFrame(factor_terminal(c, mcfg, 13, 20000, 1), columns=c['factors'] + ['RATE'])
display(Fs.describe(percentiles=[.01,.05,.5,.95]).T.round(3))
Fs[['SPY','GLD','HYG','BTC-USD']].clip(-0.6, 0.8).plot.hist(bins=80, alpha=.5, figsize=(12,4)); plt.title('13-week returns'); plt.show()

## 6. Risk of each position of today's portfolio in the scenarios

In [ ]:
from agents.modeling_agent.scenarios import asset_scenarios
tg = pd.read_csv(D/'portfolio/target_portfolio.csv', index_col=0); w = tg.weight_real/tg.weight_real.sum()
R = asset_scenarios(pe, macro, mcfg, list(w.index), pe.index[-1], n=20000)
q = np.quantile(R, .05, axis=0); cv = np.array([R[R[:,j] <= q[j], j].mean() for j in range(R.shape[1])])
port = R @ w.values; qp = np.quantile(port, .05); tail = port <= qp
contrib = (R[tail] * w.values).mean(axis=0)
t = pd.DataFrame({'weight': w, 'CVaR95_3m_asset': cv, 'contribution_to_tail_loss': contrib}).join(uni[['asset_class']])
print(f'Portfolio 3-month CVaR 95%: {port[tail].mean():.1%}')
t.sort_values('contribution_to_tail_loss').round(4)

## 7. Sensitivity: how risk changes with the model assumptions

In [ ]:
def cvar_port(mc):
    cc = calibrate(pe, macro, mc); from agents.modeling_agent import scenarios as S
    S._CACHE.clear(); RR = S.asset_scenarios(pe, macro, mc, list(w.index), pe.index[-1], n=20000)
    p = RR @ w.values; return p[p <= np.quantile(p, .05)].mean()
rows = {}
for lab, ch in [('base', {}), ('no risk premium', {'drift_shrink': 0.0}), ('full historical premium', {'drift_shrink': 1.0}),
                ('jumps x2', {'jumps': {**mcfg['jumps'], 'intensity': 1.0}}), ('no jumps', {'jumps': {**mcfg['jumps'], 'intensity': 0.0}}),
                ('crisis VIX threshold 20', {'crisis_vix': 20})]:
    rows[lab] = cvar_port({**mcfg, **ch})
pd.Series(rows).round(4).to_frame('portfolio 3-month CVaR95')